In [5]:
## data/ load_dataset.py
from datasets import load_dataset


def load_fiqa_pairs() -> list[tuple]:
    corpus = []

    print(f"Loading fiqa dataset....")
    fiqa_data = load_dataset("LLukas22/fiqa")
    for row in fiqa_data["train"]:
        question, answer = row.get("question", ""), row.get("answer", "")
        if len(question.strip()) > 10 and len(answer.strip()) > 10:
            corpus.append((question, answer))

    print(f"corpus length after loading fiqa data: {len(corpus)}")

    print(f"Loading FinGPT data....")
    fingpt_data = load_dataset("FinGPT/fingpt-fiqa_qa")
    for row in fingpt_data["train"]:
        question, answer = row.get("input", ""), row.get("output", "")
        if len(question.strip()) > 10 and len(answer.strip()) > 10:
            corpus.append((question, answer))

    print(f"corpus length after loading FinGPT data: {len(corpus)}")
    print("\nSample pairs:")
    for q, a in corpus[:3]:
        print(f"Q: {q[:100]}")
        print(f"A: {a[:100]}")
        print("---")
    
    return corpus


pairs = load_fiqa_pairs()
print(f"\n Total pairs: {len(pairs)}")

Loading fiqa dataset....


README.md: 0.00B [00:00, ?B/s]

train.json:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14511 [00:00<?, ? examples/s]

...

corpus length after loading fiqa data: 14506
Loading FinGPT data....


README.md:   0%|          | 0.00/522 [00:00<?, ?B/s]

data/train-00000-of-00001-ab79bf9300210e(…):   0%|          | 0.00/10.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17110 [00:00<?, ? examples/s]

corpus length after loading FinGPT data: 31573

Sample pairs:
Q: What do brokers do with bad stock?
A: For every seller, there's a buyer. Buyers may have any reason for wanting to buy (bargain shopping, 
---
Q: Why do investors buy stock that had appreciated?
A: You seem to prefer to trade like I do: "Buy low, sell high." But there are some people that prefer a
---
Q: Is it ever a good idea to close credit cards?
A: Yes, it can be a good idea to close unused credit cards.  I am going to give some reasons why it can
---

 Total pairs: 31573


In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.9.0+cu126
True


In [6]:
## finetuning.py
from sentence_transformers import SentenceTransformer, InputExample
from torch.utils.data import DataLoader
from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss
from torch.optim import AdamW

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
pairs = load_fiqa_pairs()

## Convert pairs to a list of InputExample objects
train_examples = [InputExample(texts=[pair[0], pair[1]]) for pair in pairs]

## Initialising the loss function
base_loss = MultipleNegativesRankingLoss(model)
loss_function = MatryoshkaLoss(model, base_loss, matryoshka_dims=[64, 128, 256, 384])

train_dataloader = DataLoader(train_examples, batch_size=32, shuffle=True, collate_fn=model.smart_batching_collate)
print(f"Total training examples: {len(train_examples)}")
print(f"Sample: {train_examples[0].texts}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading fiqa dataset....
corpus length after loading fiqa data: 14506
Loading FinGPT data....
corpus length after loading FinGPT data: 31573

Sample pairs:
Q: What do brokers do with bad stock?
A: For every seller, there's a buyer. Buyers may have any reason for wanting to buy (bargain shopping, 
---
Q: Why do investors buy stock that had appreciated?
A: You seem to prefer to trade like I do: "Buy low, sell high." But there are some people that prefer a
---
Q: Is it ever a good idea to close credit cards?
A: Yes, it can be a good idea to close unused credit cards.  I am going to give some reasons why it can
---
Total training examples: 31573
Sample: ['What do brokers do with bad stock?', "For every seller, there's a buyer. Buyers may have any reason for wanting to buy (bargain shopping, foolish belief in a crazy business, etc).  The party (brokerage, market maker, individual) owning the stock at the time the company goes out of business is the loser . But in a general panic, not every 

In [13]:
from tqdm import tqdm
EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5)
device = model.device

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    progress_bar = tqdm(
        train_dataloader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        unit="batch")

    for batch in train_dataloader:
        optimizer.zero_grad()
        features, labels = batch
        labels.to(device)
        features = [{k:v.to(device) for k, v in f.items()} for f in features]
        loss = loss_function(features, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        
        # update progress bar with current loss
        progress_bar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'avg_loss': f"{total_loss / (progress_bar.n + 1):.4f}"
        })

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_dataloader):.4f}")


Epoch 3/3:   0%|          | 0/987 [20:34<?, ?batch/s, loss=0.8821, avg_loss=888.8558]

Epoch 1/3:   0%|          | 0/987 [00:10<?, ?batch/s, loss=0.8590, avg_loss=9.3864]

KeyboardInterrupt: 

In [12]:
model.save("matryoshka-bge-small-finance")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
import shutil
shutil.make_archive("output", 'zip', "/kaggle/working/matryoshka-bge-small-finance")

'/kaggle/working/output.zip'

In [1]:
## quick test
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("/kaggle/input/models/enigmaticbrain/matryoshka-rag-trained-weight/pytorch/default/1")
test_sentences = [
    "What is EBITDA?",
    "EBITDA stands for Earnings Before Interest Taxes Depreciation and Amortisation",
    "The Federal Reserve raised interest rates",
    "What is the yield curve?"
]

embeddings = model.encode(test_sentences)

sim_matrix = cosine_similarity(embeddings)
print(np.round(sim_matrix, 2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[[1.   0.82 0.03 0.11]
 [0.82 1.   0.15 0.13]
 [0.03 0.15 1.   0.44]
 [0.11 0.13 0.44 1.  ]]


In [18]:
import time
eval_pairs = load_fiqa_pairs()[-50:]
questions = [pair[0] for pair in eval_pairs]
answers = [pair[1] for pair in eval_pairs]

## encode everything at full dimensions
q_embeddings = model.encode(questions)
a_embeddings = model.encode(answers)

def evaluate_at_dim(q_emb, a_emb, dim):
    q_emb_sliced = q_emb[:, :dim]
    a_emb_sliced = a_emb[:, :dim]

    start_time = time.perf_counter()
    similarity_matrix = cosine_similarity(q_emb_sliced, a_emb_sliced)
    latency_ms = (time.perf_counter() - start_time) * 1000
    
    count = sum(1 for idx, row in enumerate(similarity_matrix) if np.argmax(row)==idx)

    recall = count*100 / len(similarity_matrix)
    return recall, latency_ms

dims = [64, 128, 256, 384]
print(f"\n {'Dim':<8} {'Recall':>10} {'Latency (ms)':>14}")
print('-'*35)
for dim in dims:
    recall, latency = evaluate_at_dim(q_embeddings, a_embeddings, dim)
    print(f"\n {dim:<8} {recall:>9.1f}% {latency:>13.2f}ms")

Loading fiqa dataset....
corpus length after loading fiqa data: 14506
Loading FinGPT data....
corpus length after loading FinGPT data: 31573

Sample pairs:
Q: What do brokers do with bad stock?
A: For every seller, there's a buyer. Buyers may have any reason for wanting to buy (bargain shopping, 
---
Q: Why do investors buy stock that had appreciated?
A: You seem to prefer to trade like I do: "Buy low, sell high." But there are some people that prefer a
---
Q: Is it ever a good idea to close credit cards?
A: Yes, it can be a good idea to close unused credit cards.  I am going to give some reasons why it can
---

 Dim          Recall   Latency (ms)
-----------------------------------

 64            30.0%          0.99ms

 128           34.0%          0.59ms

 256           36.0%          0.87ms

 384           36.0%          0.97ms


In [19]:
eval_pairs = load_fiqa_pairs()[-5:]
questions = [pair[0] for pair in eval_pairs]
print(questions)

Loading fiqa dataset....
corpus length after loading fiqa data: 14506
Loading FinGPT data....
corpus length after loading FinGPT data: 31573

Sample pairs:
Q: What do brokers do with bad stock?
A: For every seller, there's a buyer. Buyers may have any reason for wanting to buy (bargain shopping, 
---
Q: Why do investors buy stock that had appreciated?
A: You seem to prefer to trade like I do: "Buy low, sell high." But there are some people that prefer a
---
Q: Is it ever a good idea to close credit cards?
A: Yes, it can be a good idea to close unused credit cards.  I am going to give some reasons why it can
---
['Pensions, annuities, and “retirement”', 'What does it mean to a life insurance policy holder to convert from a stock to mutual insurance company?', 'Can capital loss in traditional IRA and Roth IRA be used to offset taxable income?', 'Can capital loss in traditional IRA and Roth IRA be used to offset taxable income?', 'Selling a stock for gain to offset other stock loss']


**Last 5 questions in the dataset**

```['Pensions, annuities, and “retirement”', 'What does it mean to a life insurance policy holder to convert from a stock to mutual insurance company?', 'Can capital loss in traditional IRA and Roth IRA be used to offset taxable income?', 'Can capital loss in traditional IRA and Roth IRA be used to offset taxable income?', 'Selling a stock for gain to offset other stock loss']```

## Annotation noise
If you notice in the above last five results, two questions are identical ("Can capital loss in traditional IRA and Roth IRA be used to offset taxable income?"). This is one of the data quality issues that we see low recall@1. Both will have high cosine similarity to the same pool of answers, but for the first instance, it might return the answer of the second instance - which is semantically correct but counts as wrong in Recall@1 calculation. This is called annotation noise.

## Standard Solutions:
### Deduplication the eval set
Removing the questions with cosine similarity above a threshold to each other.

### Use Recall@3 instead of Recall@1
Count it as correct if the right answer appears in the top3 result, not just rank 1. This is more forgiving of near-duplicate confusions.



In [21]:
## Method - 1
def deduplicate_pairs(pairs, model, threshold=0.85):
    questions = [p[0] for p in pairs]
    q_emb = model.encode(questions)
    sim_matrix = cosine_similarity(q_emb)
    
    keep = []
    dropped = set()
    for i in range(len(pairs)):
        if i in dropped:
            continue
        keep.append(i)
        for j in range(i+1, len(pairs)):
            if sim_matrix[i][j] > threshold:
                dropped.add(j)
    
    return [pairs[i] for i in keep]


## Method - 2
def evaluate_at_dim(q_emb, a_emb, dim, k=3):
    q_sliced = q_emb[:, :dim]
    a_sliced = a_emb[:, :dim]
    
    runs = 100
    start = time.perf_counter()
    for _ in range(runs):
        similarity_matrix = cosine_similarity(q_sliced, a_sliced)
    latency_ms = (time.perf_counter() - start) * 1000 / runs
    
    count = 0
    for idx, row in enumerate(similarity_matrix):
        # top-k indices sorted by similarity
        top_k = np.argsort(row)[::-1][:k]
        if idx in top_k:
            count += 1
    
    recall_at_k = count * 100 / len(similarity_matrix)
    return recall_at_k, latency_ms

In [22]:
all_pairs = load_fiqa_pairs()
eval_pairs = all_pairs[100:200]

print(f"Before deduplication: {len(eval_pairs)} pairs")

eval_pairs = deduplicate_pairs(eval_pairs, model, threshold=0.85)

print(f"After deduplication: {len(eval_pairs)} pairs")
# encode
questions = [p[0] for p in eval_pairs]
answers   = [p[1] for p in eval_pairs]

q_embeddings = model.encode(questions)
a_embeddings = model.encode(answers)

# evaluate at all dims
dims = [64, 128, 256, 384]
print(f"\n{'Dim':<8} {'Recall@1':>10} {'Recall@3':>10} {'Latency(ms)':>14}")
print("-" * 45)
for dim in dims:
    r1, latency = evaluate_at_dim(q_embeddings, a_embeddings, dim, k=1)
    r3, _ = evaluate_at_dim(q_embeddings, a_embeddings, dim, k=3)
    print(f"{dim:<8} {r1:>9.1f}% {r3:>9.1f}% {latency:>13.2f}ms")

Loading fiqa dataset....
corpus length after loading fiqa data: 14506
Loading FinGPT data....
corpus length after loading FinGPT data: 31573

Sample pairs:
Q: What do brokers do with bad stock?
A: For every seller, there's a buyer. Buyers may have any reason for wanting to buy (bargain shopping, 
---
Q: Why do investors buy stock that had appreciated?
A: You seem to prefer to trade like I do: "Buy low, sell high." But there are some people that prefer a
---
Q: Is it ever a good idea to close credit cards?
A: Yes, it can be a good idea to close unused credit cards.  I am going to give some reasons why it can
---
Before deduplication: 100 pairs
After deduplication: 100 pairs

Dim        Recall@1   Recall@3    Latency(ms)
---------------------------------------------
64            54.0%      75.0%          0.51ms
128           72.0%      79.0%          0.66ms
256           74.0%      83.0%          0.83ms
384           76.0%      87.0%          0.89ms


In [26]:
all_pairs = load_fiqa_pairs()
eval_pairs = all_pairs[-1500:]

print(f"Before deduplication: {len(eval_pairs)} pairs")

eval_pairs = deduplicate_pairs(eval_pairs, model, threshold=0.85)

print(f"After deduplication: {len(eval_pairs)} pairs")
# encode
questions = [p[0] for p in eval_pairs]
answers   = [p[1] for p in eval_pairs]

q_embeddings = model.encode(questions)
a_embeddings = model.encode(answers)

# evaluate at all dims
dims = [64, 128, 256, 384]
print(f"\n{'Dim':<8} {'Recall@1':>10} {'Recall@3':>10} {'Latency(ms)':>14}")
print("-" * 45)
for dim in dims:
    r1, latency = evaluate_at_dim(q_embeddings, a_embeddings, dim, k=1)
    r3, _ = evaluate_at_dim(q_embeddings, a_embeddings, dim, k=3)
    print(f"{dim:<8} {r1:>9.1f}% {r3:>9.1f}% {latency:>13.2f}ms")

Loading fiqa dataset....
corpus length after loading fiqa data: 14506
Loading FinGPT data....
corpus length after loading FinGPT data: 31573

Sample pairs:
Q: What do brokers do with bad stock?
A: For every seller, there's a buyer. Buyers may have any reason for wanting to buy (bargain shopping, 
---
Q: Why do investors buy stock that had appreciated?
A: You seem to prefer to trade like I do: "Buy low, sell high." But there are some people that prefer a
---
Q: Is it ever a good idea to close credit cards?
A: Yes, it can be a good idea to close unused credit cards.  I am going to give some reasons why it can
---
Before deduplication: 1500 pairs
After deduplication: 611 pairs

Dim        Recall@1   Recall@3    Latency(ms)
---------------------------------------------
64            33.9%      48.0%          1.28ms
128           44.7%      62.4%          1.72ms
256           53.8%      70.0%          2.53ms
384           57.1%      73.5%          3.06ms
